# NZ Dairy Plus — AI Data & Automation System
**Author:** V P Vishal | Master of Business Analytics, University of Auckland

- **System 1:** Smart Upsell Assistant (Market Basket Analysis)
- **System 2:** Missed Opportunity Finder (Pattern Recognition)
- **System 3:** AI-Generated Personalized Messages (Claude API)

In [ ]:
# Install dependencies
!pip install anthropic pandas numpy -q

import os
# Set your API key here
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict
from itertools import combinations
import json
import os
import anthropic

CONFIG = {
    'min_support': 0.05,
    'min_confidence': 0.30,
    'frequency_threshold': 0.70,
    'recency_days': 14,
    'sample_orders': 500,
}

print("Imports complete.")

In [ ]:
def generate_sample_data(n_orders=500):
    np.random.seed(42)
    products = {
        'milk':      {'category': 'dairy', 'price': 4.50},
        'cheese':    {'category': 'dairy', 'price': 8.00},
        'cream':     {'category': 'dairy', 'price': 6.50},
        'butter':    {'category': 'dairy', 'price': 5.00},
        'yogurt':    {'category': 'dairy', 'price': 7.00},
        'sour_cream':{'category': 'dairy', 'price': 5.50},
    }
    customer_types = {
        'cafe':       ['milk', 'cream'],
        'restaurant': ['butter', 'cream', 'cheese'],
        'bakery':     ['milk', 'butter', 'cream'],
        'grocery':    ['milk', 'cheese', 'yogurt'],
    }
    orders = []
    customers = [f'CUST_{i:03d}' for i in range(1, 51)]
    for order_id in range(1, n_orders + 1):
        customer = np.random.choice(customers)
        customer_type = np.random.choice(list(customer_types.keys()))
        base_products = customer_types[customer_type]
        order_date = datetime.now() - timedelta(days=np.random.randint(0, 90))
        order_products = base_products.copy()
        if np.random.random() > 0.4:
            extra = np.random.choice(list(products.keys()))
            if extra not in order_products:
                order_products.append(extra)
        for product in order_products:
            orders.append({
                'order_id':     f'ORD_{order_id:04d}',
                'customer_id':  customer,
                'customer_type': customer_type,
                'order_date':   order_date,
                'day_of_week':  order_date.strftime('%A'),
                'product':      product,
                'quantity':     np.random.randint(1, 10),
                'price':        products[product]['price']
            })
    df = pd.DataFrame(orders)
    os.makedirs('data', exist_ok=True)
    df.to_csv('data/sample_orders.csv', index=False)
    return df

In [ ]:
class SmartUpsellAssistant:
    def __init__(self, min_support=0.05, min_confidence=0.30):
        self.min_support = min_support
        self.min_confidence = min_confidence
        self.upsell_rules = []

    def analyze_baskets(self, df):
        baskets = df.groupby('order_id')['product'].apply(list).values
        total_orders = len(baskets)
        item_counts = defaultdict(int)
        for basket in baskets:
            for item in set(basket):
                item_counts[item] += 1
        pair_counts = defaultdict(int)
        for basket in baskets:
            for pair in combinations(sorted(set(basket)), 2):
                pair_counts[pair] += 1
        rules = []
        for (item_a, item_b), count in pair_counts.items():
            support = count / total_orders
            confidence_a_to_b = count / item_counts[item_a]
            confidence_b_to_a = count / item_counts[item_b]
            lift = confidence_a_to_b / (item_counts[item_b] / total_orders)
            if support >= self.min_support and confidence_a_to_b >= self.min_confidence:
                rules.append({'if_bought': item_a, 'recommend': item_b,
                               'confidence': round(confidence_a_to_b, 3),
                               'support': round(support, 3), 'lift': round(lift, 2),
                               'co_occurrence': count})
            if support >= self.min_support and confidence_b_to_a >= self.min_confidence:
                lift_b_to_a = confidence_b_to_a / (item_counts[item_a] / total_orders)
                rules.append({'if_bought': item_b, 'recommend': item_a,
                               'confidence': round(confidence_b_to_a, 3),
                               'support': round(support, 3), 'lift': round(lift_b_to_a, 2),
                               'co_occurrence': count})
        self.upsell_rules = sorted(rules, key=lambda x: x['confidence'], reverse=True)
        os.makedirs('outputs', exist_ok=True)
        pd.DataFrame(self.upsell_rules).to_csv('outputs/upsell_rules.csv', index=False)
        return pd.DataFrame(self.upsell_rules)

    def get_recommendations(self, current_cart, top_n=3):
        recommendations = []
        for item in current_cart:
            for rule in self.upsell_rules:
                if rule['if_bought'] == item and rule['recommend'] not in current_cart:
                    recommendations.append({
                        'recommend': rule['recommend'],
                        'confidence': rule['confidence'],
                        'lift': rule['lift'],
                        'reason': f"Customers who buy {item} also buy {rule['recommend']} "
                                  f"{rule['confidence']*100:.0f}% of the time (Lift: {rule['lift']}x)"
                    })
        seen = set()
        unique_recs = []
        for rec in sorted(recommendations, key=lambda x: x['confidence'], reverse=True):
            if rec['recommend'] not in seen:
                seen.add(rec['recommend'])
                unique_recs.append(rec)
        return unique_recs[:top_n]

In [ ]:
class MissedOpportunityFinder:
    def __init__(self, frequency_threshold=0.70, recency_days=14):
        self.frequency_threshold = frequency_threshold
        self.recency_days = recency_days
        self.patterns = []

    def analyze_patterns(self, df):
        df['order_date'] = pd.to_datetime(df['order_date'])
        today = df['order_date'].max()
        patterns = []
        for customer in df['customer_id'].unique():
            cust_data = df[df['customer_id'] == customer].copy()
            for product in cust_data['product'].unique():
                prod_orders = cust_data[cust_data['product'] == product]
                order_dates = sorted(prod_orders['order_date'].unique())
                if len(order_dates) >= 3:
                    gaps = [(order_dates[i+1] - order_dates[i]).days
                            for i in range(len(order_dates)-1)]
                    avg_gap = np.mean(gaps)
                    gap_std = np.std(gaps)
                    consistency = 1 - (gap_std / avg_gap if avg_gap > 0 else 1)
                    last_order = order_dates[-1]
                    days_since = (today - last_order).days
                    expected_next = last_order + timedelta(days=avg_gap)
                    if consistency > self.frequency_threshold and days_since >= avg_gap * 0.8:
                        day_counts = prod_orders['day_of_week'].value_counts()
                        preferred_day = day_counts.index[0] if len(day_counts) > 0 else None
                        if days_since > avg_gap:
                            urgency = "HIGH - Overdue"
                        elif days_since >= avg_gap * 0.9:
                            urgency = "MEDIUM - Due soon"
                        else:
                            urgency = "LOW - On schedule"
                        patterns.append({
                            'customer_id': customer,
                            'product': product,
                            'avg_order_gap_days': round(avg_gap, 1),
                            'consistency_score': round(consistency, 2),
                            'last_order_date': last_order.strftime('%Y-%m-%d'),
                            'days_since_last': days_since,
                            'expected_next_order': expected_next.strftime('%Y-%m-%d'),
                            'preferred_day': preferred_day,
                            'total_orders': len(order_dates),
                            'avg_quantity': round(prod_orders['quantity'].mean(), 1),
                            'urgency': urgency,
                            'recommendation': self._generate_recommendation(
                                customer, product, avg_gap, preferred_day, days_since)
                        })
        self.patterns = sorted(patterns, key=lambda x: x['consistency_score'], reverse=True)
        os.makedirs('outputs', exist_ok=True)
        pd.DataFrame(self.patterns).to_csv('outputs/patterns.csv', index=False)
        return pd.DataFrame(self.patterns)

    def _generate_recommendation(self, customer, product, gap, day, overdue):
        if overdue > gap:
            return (f"URGENT: {customer} typically orders {product} every {gap:.0f} days "
                    f"on {day}. They're {overdue:.0f} days overdue. Reach out now!")
        return (f"{customer} orders {product} consistently every {gap:.0f} days "
                f"on {day}. Excellent subscription candidate.")

In [ ]:
class AutomationTrigger:
    @staticmethod
    def generate_upsell_message(customer_id, current_cart, recommendations):
        if not recommendations:
            return None
        top_rec = recommendations[0]
        message = (f"Hi {customer_id}! Based on your order "
                   f"({', '.join(current_cart)}), customers also love adding: "
                   f"{top_rec['recommend']}. Add it before your order deadline? "
                   f"Reply YES to add it now.")
        return {
            'customer_id': customer_id, 'message_type': 'upsell', 'priority': 'medium',
            'message_body': message,
            'products_to_add': [r['recommend'] for r in recommendations],
            'confidence': top_rec['confidence'], 'trigger_time': 'before_order_deadline',
            'channels': ['sms', 'email'],
            'expected_value': f"${recommendations[0].get('price', 8.00):.2f}"
        }

    @staticmethod
    def generate_subscription_prompt(pattern):
        message = (f"Hi {pattern['customer_id']}! We noticed you order "
                   f"{pattern['product']} every {pattern['avg_order_gap_days']:.0f} days "
                   f"on {pattern['preferred_day']}. Want to set up auto-delivery? "
                   f"Reply YES and we'll handle it!")
        return {
            'customer_id': pattern['customer_id'], 'message_type': 'subscription',
            'priority': pattern['urgency'].split(' - ')[0].lower(),
            'message_body': message, 'product': pattern['product'],
            'frequency_days': pattern['avg_order_gap_days'],
            'preferred_day': pattern['preferred_day'], 'quantity': pattern['avg_quantity'],
            'trigger_time': pattern['expected_next_order'],
            'channels': ['sms', 'email'], 'urgency': pattern['urgency']
        }

    @staticmethod
    def save_automation_queue(upsells, subscriptions):
        queue = {
            'generated_at': datetime.now().isoformat(),
            'upsell_triggers': upsells,
            'subscription_triggers': subscriptions,
            'total_messages': len(upsells) + len(subscriptions)
        }
        os.makedirs('outputs', exist_ok=True)
        with open('outputs/automation_queue.json', 'w') as f:
            json.dump(queue, f, indent=2)
        return queue

In [ ]:
def generate_ai_message(customer_id, product, gap_days, urgency, message_type, cart=None):
    """
    Uses Claude API to generate personalized outreach messages,
    replacing hardcoded templates with dynamic AI-written copy.
    """
    client = anthropic.Anthropic()

    if message_type == "subscription":
        prompt = (
            f"Write a friendly, conversational SMS to customer {customer_id}. "
            f"They consistently order {product} every {gap_days:.0f} days. "
            f"Urgency: {urgency}. "
            f"Suggest they set up auto-delivery. "
            f"Keep it under 160 characters. Sound human, not robotic. No emojis."
        )
    else:
        prompt = (
            f"Write a friendly 1-sentence SMS to {customer_id} "
            f"suggesting they add {product} to their current order of {', '.join(cart or [])}. "
            f"Keep it under 160 characters. Sound helpful, not pushy. No emojis."
        )

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text.strip()

In [ ]:
print("=" * 80)
print("STEP 1: Loading Order Data")
print("-" * 80)
df = generate_sample_data(n_orders=CONFIG['sample_orders'])
print(f"Loaded {len(df)} order line items")
print(f"{df['order_id'].nunique()} unique orders")
print(f"{df['customer_id'].nunique()} customers")
print(f"{df['product'].nunique()} products")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print(f"Data saved to: data/sample_orders.csv")

In [ ]:
print("=" * 80)
print("STEP 2: SMART UPSELL ASSISTANT (Market Basket Analysis)")
print("=" * 80)

upsell = SmartUpsellAssistant(
    min_support=CONFIG['min_support'],
    min_confidence=CONFIG['min_confidence']
)
rules_df = upsell.analyze_baskets(df)
print(f"Discovered {len(rules_df)} upsell rules")
print(f"Results saved to: outputs/upsell_rules.csv")
print()
print("Top 10 Strongest Associations:")
print("-" * 80)
display_cols = ['if_bought', 'recommend', 'confidence', 'support', 'lift']
print(rules_df[display_cols].head(10).to_string(index=False))
print()

demo_cart = ['milk']
recommendations = upsell.get_recommendations(demo_cart, top_n=3)
print(f"LIVE DEMO: Customer adds 'milk' to cart")
print("-" * 80)
if recommendations:
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. Recommend: {rec['recommend']} | Confidence: {rec['confidence']*100:.0f}% | Lift: {rec['lift']}x")
    trigger = AutomationTrigger.generate_upsell_message('CUST_001', demo_cart, recommendations)
    print(f"\nTemplate SMS: {trigger['message_body']}")

In [ ]:
print("=" * 80)
print("STEP 3: MISSED OPPORTUNITY FINDER (Pattern Recognition)")
print("=" * 80)

opportunity = MissedOpportunityFinder(frequency_threshold=CONFIG['frequency_threshold'])
patterns_df = opportunity.analyze_patterns(df)
print(f"Detected {len(patterns_df)} predictable patterns")
print(f"Results saved to: outputs/patterns.csv")
print()

if len(patterns_df) > 0:
    display_cols = ['customer_id', 'product', 'avg_order_gap_days',
                    'consistency_score', 'days_since_last', 'urgency']
    print("Top 10 Subscription Opportunities:")
    print("-" * 80)
    print(patterns_df[display_cols].head(10).to_string(index=False))
    print()
    top_pattern = patterns_df.iloc[0].to_dict()
    sub_trigger = AutomationTrigger.generate_subscription_prompt(top_pattern)
    print(f"Template SMS: {sub_trigger['message_body']}")

In [ ]:
print("=" * 80)
print("STEP 4: Generating Automation Queue")
print("=" * 80)

upsell_triggers = []
subscription_triggers = []

for customer in df['customer_id'].unique()[:5]:
    cust_orders = df[df['customer_id'] == customer]
    recent_products = cust_orders.nlargest(1, 'order_date')['product'].tolist()
    if recent_products:
        recs = upsell.get_recommendations(recent_products, top_n=2)
        if recs:
            trigger = AutomationTrigger.generate_upsell_message(customer, recent_products, recs)
            if trigger:
                upsell_triggers.append(trigger)

for _, pattern in patterns_df.iterrows():
    trigger = AutomationTrigger.generate_subscription_prompt(pattern.to_dict())
    subscription_triggers.append(trigger)

queue = AutomationTrigger.save_automation_queue(upsell_triggers, subscription_triggers)
print(f"Generated {len(upsell_triggers)} upsell messages")
print(f"Generated {len(subscription_triggers)} subscription prompts")
print(f"Total automation triggers: {queue['total_messages']}")
print(f"Queue saved to: outputs/automation_queue.json")

In [ ]:
print("=" * 80)
print("STEP 4.5: AI-GENERATED PERSONALIZED MESSAGES (Claude API)")
print("=" * 80)
print()
print("Generating AI messages for top subscription opportunities...")
print("-" * 80)

for i, (_, pattern) in enumerate(patterns_df.head(3).iterrows()):
    template_msg = AutomationTrigger.generate_subscription_prompt(
        pattern.to_dict())['message_body']
    ai_msg = generate_ai_message(
        customer_id=pattern['customer_id'],
        product=pattern['product'],
        gap_days=pattern['avg_order_gap_days'],
        urgency=pattern['urgency'],
        message_type="subscription"
    )
    print(f"\nCustomer: {pattern['customer_id']} | Product: {pattern['product']}")
    print(f"Template : {template_msg}")
    print(f"AI version: {ai_msg}")

print()
print("Generating AI messages for top upsell opportunities...")
print("-" * 80)

for customer in df['customer_id'].unique()[:3]:
    cust_orders = df[df['customer_id'] == customer]
    recent_products = cust_orders.nlargest(1, 'order_date')['product'].tolist()
    if recent_products:
        recs = upsell.get_recommendations(recent_products, top_n=1)
        if recs:
            template_msg = AutomationTrigger.generate_upsell_message(
                customer, recent_products, recs)['message_body']
            ai_msg = generate_ai_message(
                customer_id=customer,
                product=recs[0]['recommend'],
                gap_days=0,
                urgency="medium",
                message_type="upsell",
                cart=recent_products
            )
            print(f"\nCustomer: {customer} | Cart: {recent_products}")
            print(f"Template : {template_msg}")
            print(f"AI version: {ai_msg}")

print()
print("AI messaging layer integrated successfully")
print("Claude replaces hardcoded templates with personalized outreach")

In [ ]:
print("=" * 80)
print("STEP 5: BUSINESS IMPACT ANALYSIS")
print("=" * 80)

total_customers = df['customer_id'].nunique()
total_orders = df['order_id'].nunique()
avg_order_value = (df.groupby('order_id')['price'].sum()).mean()
weekly_upsell_revenue = total_customers * 0.20 * 8.00
monthly_recurring_revenue = len(patterns_df) * 0.30 * avg_order_value

print(f"Active customers       : {total_customers}")
print(f"Average order value    : ${avg_order_value:.2f}")
print(f"Total orders analyzed  : {total_orders}")
print()
print(f"Upsell rules discovered: {len(rules_df)}")
print(f"Subscription candidates: {len(patterns_df)}")
print(f"Messages ready to send : {queue['total_messages']}")
print()
print(f"Weekly upsell revenue  : ${weekly_upsell_revenue:.2f}")
print(f"Monthly recurring rev  : ${monthly_recurring_revenue:.2f}")
print(f"Annual revenue increase: ${(weekly_upsell_revenue*52 + monthly_recurring_revenue*12):.2f}")
print(f"Time saved             : ~8 hours/week (80% reduction in manual outreach)")
print()
print("Next steps for production:")
print("  1. Connect to real customer database (CSV / API / SQL)")
print("  2. Integrate Twilio (SMS) + SendGrid (email)")
print("  3. Setup Zapier / Make webhooks for CRM integration")
print("  4. Connect to Azure OpenAI for enterprise deployment")
print("  5. Build conversion tracking dashboard")
print()
print("=" * 80)
print("DEMO COMPLETE")
print("=" * 80)